# Train an IGNODE-compatible detector with YOLOX (Maintained Fork)

**Output:** an ONNX model + sidecars ready to upload via IGNODE's ML Factory → Models → Upload custom model.

This notebook uses the actively-maintained [pixeltable-yolox](https://github.com/pixeltable/pixeltable-yolox) Apache fork instead of the upstream Megvii YOLOX 0.3.0 (last released Aug 2022). Same architecture, same ONNX output contract (IGNODE UINF yolox_raw decode) — different install path. Cell 1 is one `pip install` instead of a tarball-extract + patch dance.

If this notebook fails install in a way the regular YOLOX notebook doesn't, fall back to `train_image_detector_yolox.ipynb`.

In [ ]:
# Step 0 — install pixeltable-yolox + dependencies. Cell ~2 min on a fresh runtime.
#
# Single `pip install` pulls pixeltable-yolox + most deps (onnx,
# onnxsim, supervision, thop, tensorboard, pycocotools — all bundled).
#
# Three extras still needed:
#   1. opencv-python-headless override — pixeltable's pyproject pins
#      opencv-python (GUI variant) which needs libxcb.so.1 on Linux.
#      -headless ships the same cv2 with no X11 deps.
#   2. onnxruntime — NOT bundled by pixeltable. Needed by Cell 9's
#      smoke-test (runs the freshly-exported ONNX through ort).
#   3. apt build-essential / g++ / python3-dev — YOLOX-family
#      architectural choice: yolox/layers/cocoeval/cocoeval.cpp is
#      lazily compiled at FIRST eval-step inside training. CXX=g++
#      env var forces torch to use g++ (the `c++` symlink may not
#      exist even with build-essential).
#
# TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 forces torch.load's weights_only
# back to False — torch 2.6+ defaults it to True which rejects
# checkpoints containing numpy._core.multiarray.scalar. Set via
# os.environ so !python subprocesses (Cells 6/7) inherit it.
import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
os.environ['CXX'] = 'g++'

!pip install -q pixeltable-yolox
!pip install -q --force-reinstall opencv-python-headless
!pip install -q onnxruntime
!apt-get install -y -q build-essential g++ python3-dev

In [ ]:
# IR-3.S.B — Settings: pick your dataset source + tune training knobs.
# Set ONE of DATASET_URL or DATASET_DIR. URL takes precedence.
# Leave both empty and the next cell will warn + stop.
#
# Supported format: COCO (.zip with annotations/{instances_train2017,
# instances_val2017}.json + train2017/ + val2017/ folders).
# Roboflow's "COCO" export ships in this exact layout — drop the link
# from Roboflow Universe directly into DATASET_URL.

# Quick test — BCCD blood-cell-count sample (7.4 MB, 3 classes):
#   DATASET_URL = 'https://raw.githubusercontent.com/IGNODE-CONNECT/ignode-collab/main/examples/datasets/bccd.coco.zip'
#
# Your own data: any public .zip download link works.
DATASET_URL = ''

# Google Drive folder path. Mount Drive separately if you want this.
DATASET_DIR = ''

# Training knobs — edit ONE cell to tune the run.
EPOCHS = 100         # 30 for a quick smoke; 300 for a real run
BATCH_SIZE = 8       # drop to 4 on a smaller GPU (T4 free Colab)

# Local scratch directory inside Colab — keep as-is.
WORKDIR = '/content/yolox-run'

In [ ]:
# IR-3.S.D — OPTIONAL Drive mount, gated so Run-all is safe.
# Only mounts Drive when the Settings cell points DATASET_DIR at a
# /content/drive path. URL customers + local-path customers skip
# this cell silently (no Drive auth popup, no Run-all stall).
import os

_needs_drive = (not DATASET_URL) and DATASET_DIR.startswith('/content/drive')
_already_mounted = os.path.ismount('/content/drive') or os.path.isdir('/content/drive/MyDrive')

if _needs_drive and not _already_mounted:
    from google.colab import drive
    drive.mount('/content/drive')
elif _needs_drive:
    print('Drive already mounted — skipping.')
else:
    print('Not using Drive (DATASET_URL set, or DATASET_DIR is a local path).')
    print('Skipping Drive mount.')

In [ ]:
# IR-3.S.B — Load dataset + normalize to pixeltable-yolox's expected
# ./datasets/COCO/ layout, then renumber category IDs to drop
# Roboflow's phantom-supercategory quirk.
import os, pathlib, shutil, zipfile, subprocess, json

if not DATASET_URL and not DATASET_DIR:
    raise SystemExit(
        '\n⚠️  Both DATASET_URL and DATASET_DIR are empty in the Settings cell.\n'
        '   Set ONE of them, then re-run this cell.'
    )

pathlib.Path(WORKDIR).mkdir(parents=True, exist_ok=True)
COCO_ROOT = pathlib.Path(WORKDIR) / 'datasets' / 'COCO'
(COCO_ROOT / 'annotations').mkdir(parents=True, exist_ok=True)
(COCO_ROOT / 'train2017').mkdir(parents=True, exist_ok=True)
(COCO_ROOT / 'val2017').mkdir(parents=True, exist_ok=True)

# 1. Get the dataset into a known location.
if DATASET_URL:
    print(f'Downloading from public URL: {DATASET_URL}')
    _archive = pathlib.Path('/content/_dataset_download')
    _archive.mkdir(parents=True, exist_ok=True)
    _dl_path = _archive / 'dataset.zip'
    subprocess.run(['curl', '-fsSL', '-o', str(_dl_path), DATASET_URL], check=True)
    print(f'Downloaded {_dl_path.stat().st_size:,} bytes — extracting…')
    _extracted = _archive / 'unpacked'
    if _extracted.exists():
        shutil.rmtree(_extracted)
    _extracted.mkdir()
    with zipfile.ZipFile(_dl_path) as zf:
        zf.extractall(_extracted)
    SOURCE_DIR = _extracted
else:
    SOURCE_DIR = pathlib.Path(DATASET_DIR)
    if not SOURCE_DIR.is_dir():
        raise SystemExit(f'❌ DATASET_DIR={DATASET_DIR!r} does not exist.')

# 2. Re-layout Roboflow's train/ + valid/ folders into the COCO layout.
_train_src = next((p for p in SOURCE_DIR.rglob('train') if p.is_dir()), None)
_valid_src = next((p for p in SOURCE_DIR.rglob('valid') if p.is_dir()), None) \
          or next((p for p in SOURCE_DIR.rglob('val')   if p.is_dir()), None)
if _train_src is None or _valid_src is None:
    raise SystemExit(
        '❌ Could not find train/ + valid/ folders in dataset. '
        'Roboflow COCO exports include both. Check your DATASET_URL.'
    )

for jpg in _train_src.glob('*.jpg'):
    shutil.copy(jpg, COCO_ROOT / 'train2017' / jpg.name)
for jpg in _valid_src.glob('*.jpg'):
    shutil.copy(jpg, COCO_ROOT / 'val2017' / jpg.name)
shutil.copy(_train_src / '_annotations.coco.json',
            COCO_ROOT / 'annotations' / 'instances_train2017.json')
shutil.copy(_valid_src / '_annotations.coco.json',
            COCO_ROOT / 'annotations' / 'instances_val2017.json')

n_train = len(list((COCO_ROOT / 'train2017').glob('*.jpg')))
n_val   = len(list((COCO_ROOT / 'val2017').glob('*.jpg')))
print(f'train2017: {n_train} images')
print(f'val2017:   {n_val} images')

# 3. Renumber category IDs — drop Roboflow's phantom supercategory at
# id=0 + remap remaining classes to 0..N-1 so they fit num_classes.
for ann_path in (COCO_ROOT / 'annotations').glob('instances_*.json'):
    data = json.loads(ann_path.read_text())
    real_cats = [c for c in data['categories'] if c.get('supercategory') != 'none']
    real_cats.sort(key=lambda c: c['id'])
    id_map = {c['id']: new for new, c in enumerate(real_cats)}
    data['categories']  = [{**c, 'id': id_map[c['id']]} for c in real_cats]
    data['annotations'] = [{**a, 'category_id': id_map[a['category_id']]} for a in data['annotations']]
    ann_path.write_text(json.dumps(data))
    print(f"{ann_path.name}: classes → {[(c['id'], c['name']) for c in data['categories']]}")

# 4. Derive NUM_CLASSES from the renumbered annotations.
_train_ann = json.loads((COCO_ROOT / 'annotations' / 'instances_train2017.json').read_text())
NUM_CLASSES = len(_train_ann['categories'])
print(f'\nNUM_CLASSES = {NUM_CLASSES}')

In [ ]:
# Step 2 — download helper script from the public ignode-collab GitHub
# mirror. One helper for the pixeltable variant:
#
#   export_to_onnx.py — custom ONNX exporter. Works around the broken
#   `yolox export_onnx` CLI in pixeltable-yolox 0.4.2. Produces opset-18
#   ONNX matching IGNODE's UINF yolox_raw contract.
HELPERS_RAW = 'https://raw.githubusercontent.com/IGNODE-CONNECT/ignode-collab/main/colab/pixeltable_yolox'
!curl -fsSL {HELPERS_RAW}/export_to_onnx.py -o {WORKDIR}/export_to_onnx.py
!ls -la {WORKDIR}/export_to_onnx.py

In [ ]:
# Step 3 — train via pixeltable-yolox's `yolox train` CLI.
#
# Notable vs Megvii notebook:
#   - Single CLI call (no train_any.py wrapper).
#   - `-D key=value` overrides each YoloxConfig attribute.
#   - `-c yolox_s` resolves to the YoloxS subclass (depth=0.33, width=0.5).
#
# Runtime detection up top — GPU is strongly preferred. We fall back to
# CPU if no CUDA device is available, but training will be 50-100x
# slower (hours instead of minutes). TPU is NOT supported because YOLOX
# uses CUDA-only primitives (torch.cuda.amp, NCCL, torch.cuda.set_device);
# porting to PyTorch-XLA isn't on the IR-3.1 roadmap.
import os, re, json, pathlib, torch

if torch.cuda.is_available():
    DEVICES = '1'
    FP16    = '--fp16'
    print(f'GPU detected: {torch.cuda.get_device_name(0)} — training with fp16.')
else:
    DEVICES = '0'
    FP16    = ''
    print('=' * 70)
    print('WARNING: No CUDA GPU detected. Falling back to CPU training.')
    print('         CPU training is MUCH slower (hours instead of minutes).')
    print()
    print('         For best results: Runtime → Change runtime type → T4 GPU')
    print('         then re-run from Cell 1.')
    print()
    print('         (TPU is not supported — YOLOX needs CUDA-only primitives.')
    print('          Interrupt this cell now if you want to switch to a GPU')
    print('          runtime instead of waiting through CPU training.)')
    print('=' * 70)

os.chdir(WORKDIR)
!yolox train -c yolox_s -b {BATCH_SIZE} -d {DEVICES} {FP16}   -D max_epoch={EPOCHS}   -D num_classes={NUM_CLASSES}   -D data_num_workers=2   -D data_dir={WORKDIR}/datasets/COCO   2>&1 | tee {WORKDIR}/train.log

# Scrape final mAP from the COCO eval table YOLOX prints during the
# per-epoch evaluation. Take the LAST occurrence (final epoch).
_log = pathlib.Path(f'{WORKDIR}/train.log').read_text()
_ap_5095 = None
_ap_50 = None
for _line in _log.splitlines():
    m = re.search(r'IoU=0\.50:0\.95\s+\|\s*area=\s*all.*?=\s*([\d.]+|-?\d+\.\d+)', _line)
    if m: _ap_5095 = float(m.group(1))
    m = re.search(r'IoU=0\.50\s+\|\s*area=\s*all.*?=\s*([\d.]+|-?\d+\.\d+)', _line)
    if m: _ap_50 = float(m.group(1))

# Persist results.json mirroring legacy notebook's shape.
_results = {
    'mAP_50':       _ap_50,
    'mAP_50_95':    _ap_5095,
    'epochs_run':   EPOCHS,
    'batch_size':   BATCH_SIZE,
    'classes':      [c['name'] for c in sorted(_train_ann['categories'], key=lambda c: c['id'])],
    'recipe':       {'optimizer': 'SGD', 'lr_scheduler': 'yoloxwarmcos',
                     'amp_fp16': bool(FP16), 'ema': True,
                     'backbone': 'yolox_s', 'trainer': 'pixeltable-yolox'},
}
pathlib.Path(f'{WORKDIR}/results.json').write_text(json.dumps(_results, indent=2))
print(f'
Results summary:')
print(f'  mAP@0.5      = {_ap_50}')
print(f'  mAP@0.5:0.95 = {_ap_5095}')
print(f'  results.json written to {WORKDIR}/results.json')


In [ ]:
# Step 4 — export to ONNX matching IGNODE's UINF yolox_raw contract.
#
# Critical contract:
#   - decode_in_inference=False (UINF host-side decodes via anchor-grid)
#   - opset 18, input "images" [batch, 3, 640, 640], output "output"
#
# We call our own export_to_onnx.py because pixeltable-yolox 0.4.2's
# built-in `yolox export_onnx` is broken (CLI subcommand not registered
# + underlying module references the deleted `yolox.exp` namespace).
import pathlib
CKPT = next(pathlib.Path(f'{WORKDIR}/out').rglob('best_ckpt.pth'), None) \
    or next(pathlib.Path(f'{WORKDIR}/out').rglob('latest_ckpt.pth'), None)
assert CKPT is not None, 'No checkpoint under ./out/ — re-run Step 3 + inspect its logs.'
OUT = f'{WORKDIR}/model.onnx'

!python {WORKDIR}/export_to_onnx.py {CKPT} {OUT} --num-classes {NUM_CLASSES} --arch yolox_s --opset 18

In [ ]:
# IR-3.S.A — Step 5: write sidecars + auto-derive CLASS_LABELS.
# DO NOT hand-type class names here. The renumber step in Cell 4
# encodes the EXACT index→name mapping YOLOX trained against. Typing
# them in a different order ships a model where every prediction's
# label is rotated.
#
# Sidecar shape MUST match production trainer's. The 4 fields that
# matter most:
#   channel_order: 'BGR'    — YOLOX uses cv2.imread
#   rescale:       'none'   — YOLOX trains on raw [0, 255]
#   mean / std:    identity — no normalization
#   postprocess.family: 'yolox_raw' — UINF host-side decode
import json, shutil, pathlib

# Read CLASS_LABELS from results.json (single source of truth, mirrors legacy).
_results = json.loads(pathlib.Path(f'{WORKDIR}/results.json').read_text())
CLASS_LABELS = list(_results['classes'])
print(f'CLASS_LABELS (from results.json):')
for i, name in enumerate(CLASS_LABELS):
    print(f'  [{i}] {name}')
print()
print('If this order surprises you, STOP and check your source dataset.')

preprocess_config = {
    'input_size':      [640, 640],
    'mean':            [0.0, 0.0, 0.0],
    'std':             [1.0, 1.0, 1.0],
    'channel_order':   'BGR',
    'image_format':    'CHW',
    'rescale':         'none',
    'resize_method':   'letterbox',
    'letterbox_color': [114, 114, 114],
    'postprocess': {
        'family':               'yolox_raw',
        'nms_required':         True,
        'confidence_threshold': 0.25,
        'nms_iou_threshold':    0.65,
    },
    '_backbone': 'yolox_s',
    '_trainer':  'pixeltable-yolox',
}

out = pathlib.Path(f'{WORKDIR}/upload-bundle')
out.mkdir(exist_ok=True)
shutil.copy(f'{WORKDIR}/model.onnx', out / 'model.onnx')
(out / 'preprocess_config.json').write_text(json.dumps(preprocess_config, indent=2))
(out / 'class_labels.json').write_text(json.dumps(CLASS_LABELS, indent=2))

print()
print('Upload bundle ready at:', out)
!ls -la {out}

In [ ]:
# IR-3.S.A — Step 5.5: ONNX smoke-test (DO NOT SKIP).
# Runs the freshly-exported YOLOX-raw ONNX on ONE val image and prints
# the top anchor's argmax class. If the prediction obviously contradicts
# the image's content, CLASS_LABELS got reordered upstream — fix
# upload-bundle/class_labels.json BEFORE uploading.
import json, pathlib, numpy as np, onnxruntime as ort
import cv2

_bundle = pathlib.Path(f'{WORKDIR}/upload-bundle')
_labels = json.loads((_bundle / 'class_labels.json').read_text())
_sess   = ort.InferenceSession(str(_bundle / 'model.onnx'), providers=['CPUExecutionProvider'])
_inp    = _sess.get_inputs()[0].name

_val_dir = pathlib.Path(f'{WORKDIR}/datasets/COCO/val2017')
_imgs = sorted(_val_dir.glob('*.jpg'))
assert _imgs, f'No val images at {_val_dir} — re-run Step 1.'
_img_path = _imgs[0]
print(f'Smoke-testing on: {_img_path.name}')

# YOLOX preprocess: BGR, letterbox 640×640, [0..255], CHW, no norm.
_bgr = cv2.imread(str(_img_path))
_h, _w = _bgr.shape[:2]
_scale = min(640 / _h, 640 / _w)
_nh, _nw = int(_h * _scale), int(_w * _scale)
_resized = cv2.resize(_bgr, (_nw, _nh), interpolation=cv2.INTER_LINEAR)
_canvas = np.full((640, 640, 3), 114, dtype=np.uint8)
_canvas[:_nh, :_nw] = _resized
_arr = _canvas.astype(np.float32).transpose(2, 0, 1)[None, ...]

# YOLOX-raw output: (1, N_anchors, 5 + num_classes).
_outs = _sess.run(None, {_inp: _arr})
_raw  = next((o for o in _outs if o.ndim == 3 and o.shape[-1] >= 5 + len(_labels)), None)
assert _raw is not None, 'Could not find a YOLOX-raw-shaped output — confirm Step 4 export.'
_obj   = 1.0 / (1.0 + np.exp(-_raw[0, :, 4]))
_cls_l = 1.0 / (1.0 + np.exp(-_raw[0, :, 5:5 + len(_labels)]))
_best  = int((_obj[:, None] * _cls_l).max(axis=1).argmax())
_best_class = int(_cls_l[_best].argmax())
_best_obj   = float(_obj[_best])
_best_score = float((_obj[_best] * _cls_l[_best]).max())
print()
print(f'Top anchor: idx={_best_class} → class_labels[{_best_class}] = {_labels[_best_class]!r}')
print(f'Objectness: {_best_obj:.3f}  Composite score: {_best_score:.3f}')
print()
print(f'Sanity check: open {_img_path.name}. If the model thinks it contains')
print(f'a {_labels[_best_class]!r} but you can see it does not, STOP and fix')
print('upload-bundle/class_labels.json before uploading.')

## Step 6 — upload to IGNODE

In your IGNODE workspace:
1. ML Factory → Models → **Upload custom model**
2. Drag the 3 files from `upload-bundle/` into the modal
3. Deploy to a UINF instance
4. Verify in Playground — boxes should land tight on your test image

If the boxes are mispositioned: usually a channel-order or class-label-order issue. The smoke test in Cell 9 is the cheap check — sanity it first.

### Differences from the regular YOLOX (Legacy) notebook
- Uses **pixeltable-yolox** (Apache fork, actively maintained 2024-2025) instead of upstream Megvii YOLOX 0.3.0.
- Same architecture, same ONNX output contract — drop-in replacement at deploy time.
- Cell 1 is one `pip install` instead of a multi-step tarball-extract + in-place file-patch dance.